# 01 · Análisis exploratorio de datos (CRISP-DM: Comprensión de los datos)

Preguntas guía de la consigna: distribución de clases, desbalance, calidad de las imágenes y variabilidad.

> Antes, ejecutar: `python -m src.main download index preprocess`

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np, cv2, matplotlib.pyplot as plt
from src import config
from src.data.segmentation import load_image, segment, overlay
idx = pd.read_csv(config.INDEX_CSV)
idx.head()

## 1. Distribución de clases por fuente y fruta
¿Hay desbalance? ¿Alguna clase existe solo en una fuente? (riesgo de *shortcut learning*, ver `docs/datos.md`)

In [ ]:
tab = pd.crosstab([idx.source, idx.fruit], idx.quality.fillna('sin_etiqueta'), margins=True)
tab

In [ ]:
ax = pd.crosstab(idx.quality, idx.source).plot.bar(figsize=(6,3), rot=0)
ax.set_ylabel('n imágenes'); ax.set_title('Imágenes por calidad y fuente')
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / 'eda_clases.svg')

## 2. Resolución y formato de las imágenes

In [ ]:
from PIL import Image
sizes = [Image.open(config.ROOT / p).size for p in idx.path]
idx['w'], idx['h'] = zip(*sizes)
idx.groupby('source')[['w','h']].describe()

## 3. Muestras visuales por clase

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for r, q in enumerate(config.QUALITY_CLASSES):
    sub = idx[idx.quality == q]
    for c in range(6):
        ax = axes[r, c]; ax.axis('off')
        if c < len(sub):
            p = sub.sample(1, random_state=c).path.iloc[0]
            ax.imshow(cv2.cvtColor(load_image(config.ROOT / p, 256), cv2.COLOR_BGR2RGB))
            ax.set_title(q, fontsize=8)
plt.tight_layout()

## 4. Calidad de la segmentación
Revisar las segmentaciones marcadas como dudosas (`seg_ok == False`) y anotar en el informe cuántas hubo y por qué.

In [ ]:
sp = pd.read_csv(config.SPLITS_CSV)
print(sp.seg_ok.value_counts())
bad = sp[~sp.seg_ok].head(6)
fig, axes = plt.subplots(1, max(len(bad),1), figsize=(14, 3))
for ax, p in zip(np.atleast_1d(axes), bad.path):
    img = load_image(config.ROOT / p); ax.imshow(cv2.cvtColor(overlay(img, segment(img)), cv2.COLOR_BGR2RGB)); ax.axis('off')

## 5. Distribución del tamaño (d_norm) por fruta
Ojo: si las fotos propias se tomaron a otra distancia, sus d_norm no son comparables con los de Kaggle.

In [ ]:
sp.boxplot(column='d_norm', by=['fruit'], figsize=(8,3), rot=45); plt.suptitle('')
plt.savefig(config.FIGURES_DIR / 'eda_tamano.svg')

## 6. Hallazgos
_(Escribir aquí 4–6 conclusiones que justifiquen las decisiones de preparación: balanceo, fondo enmascarado, resolución, etc.)_